# 6.1 多Agent系统简介 

## 从单机到集群

单个无人机的能力终究有限，而由多个无人机组成的集群或“蜂群”则能产生力量倍增的效应。
多无人机系统的出现，源于单一平台在覆盖范围、任务效率和系统鲁棒性方面的固有局限性。
协同作业的需求推动了从单机智能到集群智能的演进。

![drone_cluster.png](img/drone_cluster.png)

## 基于大模型Agent的集群协作
本章就是将基于大型语言模型（LLM）的智能体（Agent）作为这些多无人机系统的认知核心。我们将探索实现这种协作的架构模式，通过一个动手实验来验证
——您将在高保真度的AirSim模拟器中构建并部署一个多无人机智能体系统。

但任何多智能体系统，无论是无人机集群还是分布式计算网络，都必须解决一系列根本性的挑战：

- 任务分配与分解 (Task Allocation and Decomposition)：如何将一个高层级的任务目标（例如，“搜索这片公园内的失踪徒步者”）分解为一系列可执行的子任务（例如，“无人机1搜索A区”、“无人机2搜索B区”），并如何将这些子任务分配给最合适的智能体？
- 通信 (Communication)：智能体之间需要共享哪些信息？如何构建通信协议，使其既高效又无歧义？
- 去中心化决策与共识 (Decentralized Decision-Making and Consensus)：在没有一个全知全能的中央控制器的情况下，智能体如何基于不完整的信息做出决策？一个集群如何达成集体共识或调整其整体策略？
- 路径规划与冲突消解 (Path Planning and Deconfliction)：这是确保系统物理安全的关键。每架无人机都需要规划出到达其目标的路径，同时必须避免与集群中的其他无人机以及环境中的障碍物发生碰撞。


## 多智能体协同模式

将多个独立的LLM Agent连接起来，形成一个能够协同工作的系统，需要精心设计的架构。这些架构或“设计模式”决定了智能体之间的信息流和控制关系，是影响系统效率、可扩展性和复杂性的关键设计决策。

![drone_level.jpg](img/drone_level.jpg)

### 监督者-工作者模型（集中式编排）

- 概念：这是最常见也是最直观的协同模式。一个“监督者”（Supervisor）智能体接收来自人类操作员的最高层级任务目标。它负责进行宏观的任务分解和规划，然后将具体的子任务委派给多个“工作者”（Worker）智能体。每个工作者智能体通常控制一架无人机，专注于执行分配给它的具体任务（如飞往某个区域、采集数据等）。工作者完成任务后向监督者汇报结果，监督者则负责汇总信息、评估任务进展，并规划下一步的行动。
- 框架实现（以LangGraph为例）：这种模式可以非常自然地用图（Graph）结构来表示。监督者是图中的一个核心节点，它根据当前的系统状态（State）将任务路由到其他代表工作者的节点 11。在LangGraph框架中，我们可以使用一个中央路由器或条件边（Conditional Edge）逻辑来实现监督者的决策过程，决定接下来激活哪个工作者。这种架构非常适合那些可以被清晰地分解为多个并行子任务的场景，例如区域搜索或大规模测绘。

## 去中心化与层级化模型

- 概念：当任务非常复杂，或单个监督者成为性能瓶颈时，就需要更复杂的拓扑结构。

  控制权移交（Handoffs）：智能体之间可以直接传递控制权。例如，一个负责广域搜索的“侦察”智能体在发现一个可疑目标后，可以将目标的精确坐标和控制权“移交”给一个装备了高倍变焦相机的“近距离详查”智能体

  
  层级化团队（Hierarchical Teams）：一个高层监督者可以管理多个子监督者，从而形成智能体团队。例如，一个“任务总指挥”智能体负责整个行动，它下辖一个“搜索小组长”和一个“后勤小组长”。每个小组长再分别管理一组执行具体任务的工作者无人机。
  
- 框架实现（以AutoGen为例）：AutoGen框架提出的“对话驱动”范式是模拟这些动态交互的绝佳模型 。在AutoGen中，一个GroupChatManager（群聊管理器）可以扮演动态监督者的角色。它不再遵循预设的刚性流程，而是根据“对话历史”（即智能体之间的消息传递记录）来决定下一个应该“发言”（即被激活）的智能体 。这种方式允许更灵活、更具适应性的协作，更接近人类团队的工作模式。

## 多智能体程序框架选择

目前支持多Agent的程序框架非常多，这里选择当前业界领先的三大智能体框架——Microsoft AutoGen、LangChain LangGraph和CrewAI

分析其和无人机模拟仿真的结合：

# Microsoft AutoGen：对话式协作引擎

![autogen.png](img/autogen.png)


## 核心哲学
AutoGen的设计哲学根植于一个核心理念：复杂的多智能体协作可以被有效地建模为一场结构化的对话。它并非试图构建一个刚性的任务执行流水线，而是提供了一个灵活的框架，用于创建可定制的、“可对话的”（Conversable）智能体。这些智能体可以是LLM、人类用户或自动化工具的任意组合，它们通过消息传递进行交互，共同解决问题。

- **核心理念**：复杂的多智能体协作可以被有效地建模为一场结构化的对话。
- **设计目标**：提供一个灵活的框架，用于创建可定制的、“可对话的”智能体。
- **智能体类型**：LLM、人类用户或自动化工具的任意组合。
- **交互方式**：通过消息传递进行交互，共同解决问题。

## 通信模型
AutoGen最核心的通信机制是“群聊”（Group Chat）。一个GroupChatManager（群聊管理器）负责协调多个智能体之间的对话。其关键职责是根据当前的对话历史和上下文，决定下一个应该发言的智能体。这种机制使得AutoGen能够支持高度动态和灵活的交互模式，包括：

### 顺序聊天（Sequential Chat）
任务在一个智能体链条中依次传递和处理。

### 嵌套聊天（Nested Chat）
一个复杂的工作流可以被封装成一个“子聊天”，作为一个独立的单元被其他智能体调用。

### 分层聊天（Hierarchical Chat）
智能体之间形成管理层级，例如一个“经理”智能体负责协调多个“员工”智能体的工作。

## 优势
AutoGen在处理那些解决方案路径不明确、需要通过反复试错和头脑风暴来探索的任务时，表现得尤为出色。其对话驱动的性质天然地支持迭代式推理，智能体可以相互辩论、修正彼此的观点，并逐步逼近最优解。此外，AutoGen对人类在环的无缝支持，使其非常适合用于研究探索和需要人类专家进行关键决策的复杂场景。

- **适应性**：特别适合处理解决方案路径不明确的任务。
- **迭代式推理**：支持智能体间的相互辩论和观点修正。
- **人类在环**：无缝支持人类专家参与关键决策。

链接：https://www.microsoft.com/en-us/research/project/autogen/

# LangChain LangGraph：状态驱动的编排框架

![agent_workflow.avif](img/agent_workflow.avif)


## 核心哲学
LangGraph 是构建在强大 LangChain 生态之上的一个组件，它引入了一种更为结构化和确定性的多智能体编排方式。其核心哲学是将复杂的工作流建模为一个**有状态的图**（Stateful Graph）。  
- 在该图中，**节点**（Nodes）代表智能体或普通函数。  
- **边**（Edges）定义节点之间的交互逻辑与控制流。  
- 这种方法为开发者提供了对工作流程的显式、细粒度控制，强调可预测性和工程化设计。

## 通信模型
在 LangGraph 中，智能体之间的通信主要通过一个**共享的、持久化的状态对象**（State Object）进行。  
- 每个节点执行完成后会更新该状态。  
- 后续节点根据状态的变化决定自身行为，实现数据与控制的解耦。  

控制流主要通过两种模式管理：

### 1. 工具调用（Tool Calling）
- **模式类型**：中心化控制。  
- **核心角色**：“主管”（Supervisor）智能体作为流程的中心控制器。  
- **工作方式**：主管将其他专业智能体视为可调用的“工具”，负责：  
  - 接收任务  
  - 决定调用哪个工具（即子智能体）  
  - 传递参数并整合返回结果  
- 适用于需要集中决策、流程明确的场景。

### 2. 交接（Handoffs）
- **模式类型**：去中心化控制。  
- **工作方式**：当前活动的智能体根据任务进展，主动将控制权“交接”给更合适的下一个智能体。  
- **效果**：系统活动焦点随之转移，用户可直接与新的活动智能体交互。  
- 适用于动态任务流转、角色切换频繁的复杂协作流程。

## 优势
LangGraph 非常适合构建**高度结构化、模块化、可观测**的多智能体系统，其优势体现在：

- ✅ **结构化控制流**：支持复杂分支、条件判断与循环，适用于自适应工作流。  
- ✅ **高可预测性与可追溯性**：基于图的状态机模型便于调试、监控和审计。  
- ✅ **生产级可靠性**：确定性执行机制使其在需要高可靠性和可扩展性的工业场景中表现突出。  
- ✅ **模块化设计**：节点可复用、可替换，便于系统迭代与维护。  

> 总结：LangGraph 通过“状态+图”的范式，为复杂 Agent 系统提供了工程级的构建能力，是构建生产级应用的理想选择。

链接： https://www.langchain.com/langgraph

# CrewAI：基于角色的任务执行框架

![crewai.png](img/crewai.png)


![crewai2.png](img/crewai2.png)

## 核心哲学
CrewAI旨在通过一个更高层次的抽象来简化多智能体应用的开发。它的核心哲学是将智能体协作类比为一个现实世界中的工作团队或公司部门。开发者不再需要过多关注底层的消息传递和状态管理，而是专注于定义每个智能体的角色（Role）、目标（Goal）和背景故事（Backstory）。框架本身则负责编排这些角色化的智能体，使其像一个真正的“团队”（Crew）一样协同工作。

- **核心理念**：智能体协作类比为现实世界的工作团队或公司部门。
- **开发者职责**：定义智能体的角色、目标和背景故事。
- **框架职责**：编排角色化的智能体，实现高效的团队协作。

## 通信模型
CrewAI中的通信是结构化和目标驱动的，通常遵循在“团队”（Crew）定义中指定的顺序（Sequential）或并行（Parallel）流程。任务委托（Task Delegation）是其一个核心特性，允许一个智能体将一个复杂的任务分解，并将子任务分配给其他更专业的智能体去完成。整个协作过程更像一个定义清晰的业务流程，而非一场开放式的讨论。

### 任务委托（Task Delegation）
- **描述**：一个智能体可以将复杂任务分解，并将子任务分配给其他更专业的智能体。
- **流程类型**：
  - **顺序流程（Sequential）**：任务按顺序依次传递。
  - **并行流程（Parallel）**：多个子任务同时进行。

## 优势
CrewAI最大的优势在于其直观的设计和较低的学习曲线，这使得它成为快速原型设计的绝佳工具。对于那些可以清晰地映射到“团队协作”模型的、具有明确步骤的任务（例如，研究员收集资料 -> 分析师整理数据 -> 作家撰写报告），CrewAI能够极大地简化开发过程，让开发者专注于业务逻辑而非技术细节。

- **直观设计**：易于理解和使用，适合快速原型设计。
- **低学习曲线**：开发者不需要深入理解底层机制。
- **业务逻辑聚焦**：开发者可以专注于任务的具体步骤和业务逻辑。
- **适用场景**：适用于具有明确步骤的任务，如研究、数据分析和报告撰写等。

> 总结：CrewAI通过角色化和任务委托的方式，提供了一个高效且直观的多智能体协作框架，特别适合需要清晰步骤和团队协作的任务。

链接：https://www.crewai.com/

## 无人机集群应用分析

在无人机集群应用方面，各个框架的优势分析如下：

- LangGraph 提供了最强的控制力和可预测性。如果任务是构建一个用于生产环境的、具有确定性流程的无人机自动化系统，它将是首选。然而，其基于状态机的交互模式与用户明确提出的“对话式”协同存在理念上的偏差。它能实现“协同”，但难以实现“对话”。
- CrewAI 提供了最简单的开发体验。对于快速验证一个基于角色的无人机团队（如“侦察员”->“分析员”）协作流程，它非常高效。但其高层抽象和相对固定的流程模式，可能难以支持在复杂动态环境中需要即时、灵活协商的真实无人机任务。
- AutoGen 在与用户核心需求的契合度上拥有无可比拟的优势。其整个框架都建立在“对话”这一隐喻之上，这使得它成为模拟和实现通过沟通达成协作的智能体系统的天然选择。无人机集群在未知环境中执行任务时，必然会遇到需要动态协商、调整计划的情况——例如，一架无人机发现意外障碍物，需要通过对话通知其他无人机并共同规划绕行路线。AutoGen的动态、涌现式GroupChat模型，正是为了解决这类问题而设计的。


可以看到，如果是基本的多无人机应用开发，使用LangGraph或者CrewAI的显式图编排更为简单，而如果是设计复杂完整的应用，则可以选择AutoGen。

对于一个侧重于任务执行的无人机课程而言，LangGraph的确定性控制是更佳的教学选择，但理解AutoGen所代表的灵活性范式，对于把握未来更自主化团队的发展至关重要。

这部分课程，以教务为目标，因此使用LangGraph为主，设计一个多无人机Agent场景，并基于Airsim进行实现。